# Mersenne Prime Cryptosystem

## Introduction

### 1. Mathematical Definitions

### Mersenne Numbers
A Mersenne number $M_n$ is defined by the following expression:
$$M_n = 2^n - 1$$
where $n$ is a positive integer.

* **Mersenne Prime:** A Mersenne number $M_n$ is considered a *Mersenne Prime* if the resulting value is a prime number.

### Hamming Weight
The Hamming Weight of an $n$-bit string $y$, denoted by $\text{Ham}(y)$, represents the total count of '1's in the string.

---

### 2. Complement Property
For any value $A$ (where $A \neq 0^n$), the relationship between the weight of a number and its negative (in two's complement) is given by:

$$\text{Ham}(-A) = n - \text{Ham}(A)$$



## Bit by Bit encryption

### Encrypt a single bit

In [8]:
import math
import secrets
from scipy.special import comb

def low_hamming_weight_number(n, h):
    """Gera um número de n bits com peso de Hamming h."""
    indices = set()
    while len(indices) < h:
        indices.add(secrets.randbelow(n))
    num = 0
    for i in indices:
        num |= (1 << i)
    return num

def hamming_weight(x):
    """Calcula o Peso de Hamming (quantidade de bits '1')"""
    return bin(x).count('1')

def get_n_h(lambda_param):
    """
    Finds valid n and h such that comb(n, h) >= 2^lambda
    and 4h^2 < n < 16h^2.
    """
    # Start with a reasonable n based on lambda, e.g., n approx 2*lambda
    n = int(lambda_param * 2) 
    while True:
        # Based on 4h^2 < n < 16h^2:
        # h > sqrt(n/16) and h < sqrt(n/4)
        h_min = math.floor(math.sqrt(n / 16)) + 1
        h_max = math.ceil(math.sqrt(n / 4)) - 1
        
        for h in range(h_min, h_max + 1):
            if comb(n, h, exact=True) >= 2**lambda_param:
                return n, h
        n += 1 # Increase n if no h is found for current n

# Your validated list of Mersenne exponents
MERSENNE_EXPONENTS = [
    2, 3, 5, 7, 13, 17, 19, 31, 61, 89, 107, 127, 521, 607, 1279, 2203, 
    2281, 3217, 4253, 4423, 9689, 9941, 11213, 19937, 21701, 23209, 44497, 
    86243, 110503, 132049, 216091, 756839, 859433, 1257787, 1398269, 
    2976221, 3021377, 6972593, 13466917, 20996011, 24036583, 25964951, 
    30402457, 32582657, 37156667, 42643801, 43112609, 57885161, 74207281, 
    77232917, 82589933, 136279841
]

def get_n_h_from_list(lambda_param):
    """
    Finds n from MERSENNE_EXPONENTS and h such that comb(n, h) >= 2^lambda
    and 4h^2 < n < 16h^2.
    """
    for n in MERSENNE_EXPONENTS:
        # Constraints: h > sqrt(n/16) and h < sqrt(n/4)
        h_min = math.floor(math.sqrt(n / 16)) + 1
        h_max = math.ceil(math.sqrt(n / 4)) - 1
        
        for h in range(h_min, h_max + 1):
            if comb(n, h, exact=True) >= 2**lambda_param:
                return n, h
    raise ValueError("No suitable n and h found for the given lambda.")

def generate_keys(lambda_param):
    # Retrieve n from our list and calculate corresponding h
    n, h = get_n_h_from_list(lambda_param)
    p = (2**n) - 1
    
    #generate F and G until G is invertible mod p and F, G have low Hamming weight (h)
    while True:
        F = low_hamming_weight_number(n, h)
        G = low_hamming_weight_number(n, h)
        
        try:
            # H = F * G^-1 (mod p)
            inv_G = pow(G, -1, p)
            H = (F * inv_G) % p
            return (H, G, n, h, p)
        except ValueError:
            continue # G is not invertible, try again

def encrypt(pk_H, b, n, p, h):
    """
    C = (-1)^b * (A * H + B) mod p
    """
    #Generate A and B with low Hamming weight (h)
    A = low_hamming_weight_number(n, h)
    B = low_hamming_weight_number(n, h)
    
    val = (A * pk_H + B) % p
    
    if b == 0:
        return val
    else:
        # (-1) * val no mundo modular é p - val
        return (p - val) % p

def decrypt(C, sk_G, n, p, h):
    """
    Computes d=Ham(C * G mod p) and decides based on d.
    If d <= 2h^2, output 0; if d >= n - 2h^2, output 1; else output "?".
    """
    # d = Ham(C * G mod p)
    target = (C * sk_G) % p
    d = hamming_weight(target)
    
    threshold = 2 * (h**2)
    
    if d <= threshold:
        return 0
    elif d >= n - threshold:
        return 1
    else:
        return None # Caso "?" (Indeterminado)

# --- Exemplo de Execução ---

# 1. Parâmetros (p deve ser um primo de Mersenne)
lambda_param= 256

# 2. Setup de Chaves
pk, sk, n, h, p  = generate_keys(256)

# 3. Teste com bit 0
bit_original = 1
criptograma = encrypt(pk, bit_original, n, p, h)
bit_recuperado = decrypt(criptograma, sk, n, p, h)

print(f"Parâmetros: n={n}, h={h}")
print(f"Bit Original: {bit_original}")
print(f"Criptograma (C): {hex(criptograma)[:20]}...")
print(f"Bit Recuperado: {bit_recuperado}")

Parâmetros: n=4253, h=31
Bit Original: 1
Criptograma (C): 0x1e8eeea9f109261361...
Bit Recuperado: 1


### Encrypt a message

In [9]:

def iterar(m, pk, n, p, h):
    """Encripta uma mensagem (inteiro) bit a bit."""
    mens = bin(m)[2:]
    # Usamos as funções já definidas: encrypt(pk_H, b, n, p, h)
    lista_cripto = [encrypt(pk, int(bit), n, p, h) for bit in mens]
    return lista_cripto

def iterar_descriptar(criptograma, G, n, p, h):
    """Descripta uma lista de criptogramas bit a bit."""
    # Usamos a função já definida: decrypt(C, sk_G, n, p, h)
    resultado = [decrypt(c, G, n, p, h) for c in criptograma]
    return resultado

# --- Exemplo de Execução ---

# --- Configuração Inicial ---
lambda_param = 256
pk, sk, n, h, p = generate_keys(lambda_param)


# 1. Definir a mensagem (em binário)
mensagem_str = "1101101010110101011001010101011100001011011011"
mensagem = int(mensagem_str, 2)

print(f"Mensagem original: {mensagem_str}")
print(f"Parâmetros: n={n}, h={h}")

# 2. Encriptar
lista_encriptada = iterar(mensagem, pk, n, p, h)
print(f"Criptograma gerado (primeiros 5 elementos): {lista_encriptada[:5]}...")

# 3. Descriptar
resultado = iterar_descriptar(lista_encriptada, sk, n, p, h)

# 4. Resultado final
# Filtramos os 'None' caso algum bit tenha sido indeterminado
mensagem_recuperada = ''.join(map(str, [x for x in resultado if x is not None]))
print(f"Mensagem recuperada: {mensagem_recuperada}")

# Verificação de sucesso
if mensagem_str == mensagem_recuperada:
    print("\nSucesso: A mensagem foi recuperada corretamente!")
else:
    print("\nErro: A mensagem recuperada difere da original.")

Mensagem original: 1101101010110101011001010101011100001011011011
Parâmetros: n=4253, h=31
Criptograma gerado (primeiros 5 elementos): [693943266560081595847409775703521630800911223080337988276728864527714002708685564653807835358490094455939413136313046523783927461566581417057858780019034990033793459779374894788235344040688886310472788508138811954493251074194867777527786316570269710676332483626056597201434674047025710665981092308744772393082579262427927989708163478161512149812266089477191765608115145415325274315506435609898088709992782015759854390482395183103326930722004614900393059942172146544426050881596036563383488664684446209611685642025059442167906210591466579884363516840879610020475332752604980155554553068421482098282940723745932267483887185163951960701800426150799647352195303539693988872166434873006850487602645383899662221192325945007148119656333999516476886716156962060670812832241265568873104635422528594437534915722574642704275250692927606465978883067561504249429461905914621024

### Attacking this encryption scheme

In [97]:
# 

def generate_WEAK_keys_for_attack(n, h, p):
    """The generated keys F and G will have bits only in the lower half to test the attack."""
    limite_bits = n // 2
    
    while True:
        # F e G com bits apenas na metade inferior (fator de redução da chave)
        F = low_hamming_weight_number(n, h) & ((1 << limite_bits) - 1)
        G = low_hamming_weight_number(n, h) & ((1 << limite_bits) - 1)
        
        if G == 0: continue
        try:
            inv_G = pow(G, -1, p)
            H = (F * inv_G) % p
            return (H, G, F, h)
        except ValueError:
            continue

def run_attack(H, p, F_secreto, sk_G):
    from sympy import Rational, continued_fraction_iterator, continued_fraction_convergents
    
    convergentes_encontrados = []
    fração_exata = Rational(H, p)
    cf = continued_fraction_iterator(fração_exata)
    
    # O ataque percorre os convergentes da fração contínua de H/p
    for conv in continued_fraction_convergents(cf):
        cand_G = int(conv.q)
        if cand_G == 0: continue
            
        cand_F = (H * cand_G) % p
        
        # Testar o par encontrado
        if cand_F == F_secreto and cand_G == sk_G:
            return cand_F, cand_G
            
    return None, None


# --- [1] CONFIGURAÇÃO E SETUP ---
lambda_param = 10 #Security parameter
n, h = get_n_h_from_list(lambda_param)
p = (2**n) - 1

print(f"--- [1] SETUP DO SISTEMA (lambda={lambda_param}) ---")
print(f"Parâmetros selecionados: n={n}, h={h}")

pk_H, sk_G, F_secreto, _ = generate_WEAK_keys_for_attack(n, h, p)

print(f"Chave Secreta G: {hex(sk_G)}")
print(f"Chave Pública H: {hex(pk_H)[:30]}...")

# --- [2] Starting Continuous Fractions Attack ---
print("\n--- [2] Starting Continuous Fractions Attack ---")

count=0
while count<500:
    F_atk, G_atk = run_attack(pk_H, p, F_secreto, sk_G)

    if F_atk:
        print(f"\n[✓] ATAQUE BEM-SUCEDIDO!")
        print(f"F recuperado: {hex(F_atk)}")
        print(f"G recuperado: {hex(G_atk)}")
        print(f"Iteração {count+1} concluída.")
        break
    else:
        count += 1


--- [1] SETUP DO SISTEMA (lambda=10) ---
Parâmetros selecionados: n=61, h=2
Chave Secreta G: 0x1
Chave Pública H: 0x800000...

--- [2] Starting Continuous Fractions Attack ---

[✓] ATAQUE BEM-SUCEDIDO!
F recuperado: 0x800000
G recuperado: 0x1
Iteração 1 concluída.


## Secure Public Key Cruptosystem